|<h2>Course:</h2>|<h1><a href="https://derivingsystems.com/course.html" target="_blank">Build your own vLLM: inference engines from the memory system up</a></h1>|
|-|:-:|
|<h2>Part 0:</h2>|<h1>From a Program to a Model<h1>|
|<h2>Section:</h2>|<h1>The model<h1>|
|<h2>Lecture:</h2>|<h1><b>A model is a function<b></h1>|

<br>

<h5><b>Course repo:</b> <a href="https://github.com/Venugopalan2610/vllm-from-scratch" target="_blank">github.com/Venugopalan2610/vllm-from-scratch</a></h5>
<h5><b>The derivations:</b> <a href="https://derivingsystems.com" target="_blank">derivingsystems.com</a></h5>
<i>The notebooks build the intuition. The ladder in app/ makes you build the thing.</i>

In [1]:
# Find the repo root. The directory you start from does not matter.
import sys
from pathlib import Path
ROOT = next(folder for folder in [Path.cwd(), *Path.cwd().parents]
            if (folder/'cudalib').is_dir())
sys.path.insert(0, str(ROOT))

import numpy as np
import torch
import matplotlib.pyplot as plt
from transformers import AutoModelForCausalLM, AutoTokenizer

import matplotlib_inline.backend_inline
matplotlib_inline.backend_inline.set_matplotlib_formats('svg')

# The notebook runs on a GPU if there is one, and on the CPU if not.
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
DTYPE = torch.bfloat16 if DEVICE == 'cuda' else torch.float32
MODEL_NAME = 'Qwen/Qwen3-1.7B'

/home/venugopalan/vllm-from-scratch/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# A program that you did not write

You know how to read a program. It has functions, loops and branches, and its
behavior is in the source code.

A language **model** is also a program. But you cannot read its behavior in
the source. The source is short, and it is the same for many models. The behavior
is in a large array of numbers.

This notebook opens that program and names each part. At the end you know the
words that each later lecture uses: **token**, **vocabulary**, **logits**,
**softmax**, **layer**, **hidden state** and **weights**. The
[glossary](../../GLOSSARY.md) defines each word again, if you forget one.

### Step 1: text becomes integers

A model does not read text. It reads integers.

A **tokenizer** divides the text into pieces, and gives each piece an integer
id. One piece is a **token**. The list of all the tokens that the tokenizer
knows is its **vocabulary**. An id is an index into the vocabulary.

A software analogy: the tokenizer is a codec with a fixed dictionary, and a
token id is an interned string.

In [2]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
text = 'Inference engines are fun to build.'
token_ids = tokenizer(text).input_ids
pieces = tokenizer.convert_ids_to_tokens(token_ids)

print(f'vocabulary: {len(tokenizer):,} tokens')
print(f'text:       {text!r}')
print(f'token ids:  {token_ids}')
print(f'pieces:     {pieces}')

vocabulary: 151,669 tokens
text:       'Inference engines are fun to build.'
token ids:  [641, 2202, 21106, 525, 2464, 311, 1936, 13]
pieces:     ['In', 'ference', 'Ġengines', 'Ġare', 'Ġfun', 'Ġto', 'Ġbuild', '.']


Look at the pieces. The character `Ġ` is a space. The space belongs to the
start of the next word.

A common word is one token. A rare word becomes several tokens. Try it on
words of your own.

Some pieces look strange, for example `Ã¶` in "Schrödinger". The tokenizer
works on the bytes of the text, not on characters. `ö` is two bytes, and the
display shows each byte as one symbol. So one character can need two tokens.
Part 5 `3_detokenize/` handles that problem.

In [3]:
for word in ['the', 'tokenization', 'vLLM', 'Schrödinger', 'naïve', '🙂']:
  word_ids = tokenizer(word).input_ids
  print(f'{word!r:>16} -> {len(word_ids)} tokens  {tokenizer.convert_ids_to_tokens(word_ids)}')

           'the' -> 1 tokens  ['the']
  'tokenization' -> 2 tokens  ['token', 'ization']
          'vLLM' -> 3 tokens  ['v', 'LL', 'M']
   'Schrödinger' -> 5 tokens  ['S', 'chr', 'Ã¶', 'd', 'inger']
         'naïve' -> 3 tokens  ['na', 'Ã¯', 've']
             '🙂' -> 1 tokens  ['ðŁĻĤ']


### Step 2: call the function

Now load the **model**. One call of the model is a **forward pass**. It takes
a list of token ids, and it returns a list of scores for each position.

Each position gets one score for each token of the vocabulary. These raw
scores are the **logits**. A high logit means that the model thinks this token
is a likely NEXT token.

In [4]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, dtype=DTYPE).to(DEVICE).eval()
print(f'{MODEL_NAME} on {DEVICE}')

input_ids = torch.tensor([token_ids], device=DEVICE)
with torch.no_grad():
  logits = model(input_ids).logits
print(f'input:  {tuple(input_ids.shape)}   (1 sequence, {len(token_ids)} tokens)')
print(f'output: {tuple(logits.shape)}   (one row of {logits.shape[-1]:,} scores for each position)')

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 311/311 [00:00<00:00, 14850.90it/s]

Qwen/Qwen3-1.7B on cuda


input:  (1, 8)   (1 sequence, 8 tokens)
output: (1, 8, 151936)   (one row of 151,936 scores for each position)


Row *i* of the output predicts the token at position *i + 1*. To continue
the text, you need only the last row. Look at the five tokens with the
highest logits there.

In [5]:
def top_tokens(scores, count=5):
  """-> the `count` highest scores, as (token text, score) pairs."""
  values, ids = torch.topk(scores.float(), count)
  return [(tokenizer.decode([token_id]), value.item()) for token_id, value in zip(ids, values)]

last_row = logits[0, -1]
print(f'after {text!r}, the model scores:')
for token_text, logit in top_tokens(last_row):
  print(f'  {token_text!r:>12}  logit {logit:6.2f}')

after 'Inference engines are fun to build.', the model scores:
        ' But'  logit  22.88
       ' They'  logit  22.88
         ' In'  logit  21.88
        ' The'  logit  21.62
          ' I'  logit  21.50


### Step 3: scores become probabilities

A logit has no fixed scale. **Softmax** turns the logits into probabilities.
It makes each score positive with `exp`, then it divides by the sum. The
results add up to 1.

Write it yourself in two lines, and compare it with the library version.

In [6]:
def softmax(scores):
  exponentials = torch.exp(scores - scores.max())   # subtract the max: exp cannot overflow
  return exponentials / exponentials.sum()

probs = softmax(last_row.float())
print(f'the same as torch.softmax: {torch.allclose(probs, torch.softmax(last_row.float(), -1))}')
print(f'the sum of all probabilities: {probs.sum():.4f}\n')
for token_text, prob in top_tokens(probs):
  print(f'  {token_text!r:>12}  {100*prob:5.1f}%')

the same as torch.softmax: True
the sum of all probabilities: 1.0000

        ' But'   17.4%
       ' They'   17.4%
         ' In'    6.4%
        ' The'    5.0%
          ' I'    4.4%


### Step 4: what is inside the function

The model is a stack of **layers**. All layers have the same shape, and each
layer has its own numbers.

1. The first step is a table lookup, the **embedding**. It replaces each token
   id with one row of a table. That row is a vector of numbers.
2. That vector is the **hidden state** of the token. It moves through the
   layers. Each layer reads it and adds to it.
3. Each layer has two steps. **Attention** lets a token read information from
   earlier tokens. The next section of Part 0 explains it. The **MLP** then
   transforms each token alone.
4. At the end, one large **matmul** (a matrix multiplication) turns the hidden
   state of each token into its logits.

This design is the **transformer**. Look at it in the real model.

In [7]:
config = model.config
print(f'layers:      {config.num_hidden_layers}')
print(f'hidden size: {config.hidden_size}   (the length of the vector for one token)')
print(f'vocabulary:  {config.vocab_size:,}   (a little more than the tokenizer uses: '
      f'the table size is a round number)\n')

print('one layer holds:')
for name, part in model.model.layers[0].named_children():
  print(f'  {name:<26} {type(part).__name__}')

layers:      28
hidden size: 2048   (the length of the vector for one token)
vocabulary:  151,936   (a little more than the tokenizer uses: the table size is a round number)

one layer holds:
  self_attn                  Qwen3Attention
  mlp                        Qwen3MLP
  input_layernorm            Qwen3RMSNorm
  post_attention_layernorm   Qwen3RMSNorm


In [8]:
# The embedding is only a table lookup: row number = token id.
embedding_table = model.model.embed_tokens.weight
with torch.no_grad():
  embedded = model.model.embed_tokens(input_ids)[0]
print(f'embedding table: {tuple(embedding_table.shape)}  (one row for each token of the vocabulary)')
print(f'embedded text:   {tuple(embedded.shape)}  (one row for each token of the text)')
print(f'the same as table[token_ids]: {torch.equal(embedded, embedding_table[token_ids])}')

embedding table: (151936, 2048)  (one row for each token of the vocabulary)
embedded text:   (8, 2048)  (one row for each token of the text)
the same as table[token_ids]: True


### Step 5: the weights

All the numbers in those tables and matrices are the **weights** of the model.
Another name is the **parameters**. Training set them. **Inference** is the use
of the trained model to make outputs. At inference nothing changes the
weights: they are read-only data.

The **dtype** of a number is its format. On a
[GPU](../../GLOSSARY.md#gpu) this model uses bf16: 2 bytes for each number. fp32 uses 4 bytes. Count the weights, and their size
in bytes.

In [9]:
def count(module):
  return sum(weight.numel() for weight in module.parameters())

total = count(model)
first_layer = model.model.layers[0]
parts = {
  'embedding table':        count(model.model.embed_tokens),
  'attention, all layers':  count(first_layer.self_attn) * config.num_hidden_layers,
  'MLP, all layers':        count(first_layer.mlp) * config.num_hidden_layers,
}
print(f'{total/1e9:.2f} billion weights = {total*2/1e9:.2f} GB in bf16\n')
for name, size in parts.items():
  print(f'  {name:<23} {size/1e6:7.1f} M  ({100*size/total:4.1f}%)')

1.72 billion weights = 3.44 GB in bf16

  embedding table           311.2 M  (18.1%)
  attention, all layers     352.3 M  (20.5%)
  MLP, all layers          1057.0 M  (61.4%)


### What to remember from this notebook

- A **model** is a function. Token ids go in. **Logits** come out: one score
  for each token of the **vocabulary**, at each position.
- **Softmax** turns the logits of the last position into the probabilities of
  the next token.
- Inside, each token is a vector, its **hidden state**. It goes through a
  stack of **layers**. Each layer is **attention**, then an **MLP**.
- The **weights** are a large read-only table. The model is 1.7 billion
  numbers and 3.4 GB.

One more fact is the most important fact in this course: **each forward pass
reads all the weights.** Every matrix in every layer takes part in each call.
Keep that fact. Part 1 builds on it.

The next notebook calls this function in a loop, and makes text.